<a href="https://colab.research.google.com/github/Text-Machine/mask-predict/blob/main/example_analysis_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

In [ ]:
!gdown --folder 1DEtOJg4NmJok40MHje68N-K-JPPm1bfD
!mv /content/ExperimentsData/* .
!rm -r /content/ExperimentsData

In [ ]:
!git clone https://github.com/Text-Machine/mask-predict.git

In [ ]:
%cd mask-predict

In [ ]:
!pip install -q -e .

In [1]:
import pandas as pd
import json
from tqdm import tqdm
from explain import *
from pathlib import Path

## A note on data, variable names and terminology

### A contrastive analysis masked tokens

What we pursue is **contrastive analysis** of two different concepts, captured by two distinct set of tokens. For example, the concept 'machine' is captured by 'machine' and 'machines', the contrastive concept is 'slave'.
- Note: I suggest to replace the bidrectional or reverse masking with contrastive analysis.

### Axis 1: Target vs. Contrastive Concept

One of the concepts is the **"target concept"**, to other one the **"contrastive concept"**. Each analysis starts with a target concept or token, or `TargetMaskedToken`, this is the starting point of the analysis, for example the word 'machine'. This tokens has two associated datasets. 
    - a TargetMaskedToken maps onto a set of workds that capture the target concept
    - sentences which contain the masked token (`df_target_sent`)
    - contribution obtained by from the integrated gradients method (abbr "ig", `df_target_ig`) 

The `TargetMaskedToken` related to a `ContrastiveMaskedToken`, in this case the word slave. Similary, the contrastive concepts is associated with a set of sentences that contain the tokens expressing the target concept.


### Axis 2: Observed vs. Counterfactual Language Use

To understand language model predictions, we study both the `TargetMaskedToken` and the `ContrastiveMaskedToken` in two scenarios: **observed** and **counterfactual**. 

 Observed refers to scenarios in which the target of constrastive token appears in the sentences. We than look what tokens contribute to predicting the actual word use. In counterfactual scenarios, we 

 For more target and constrastive concepts map **inversely** to observed and counterfactual tokens, i.e. when the `target` is machine, and the contrastive concept is `slave`, then when analysis machine sentences, the observed tokens are 'machine' and 'machines' and the counterfactual ones are 'slave' and 'slaves'.


In [2]:
collection,genre_suffix = 'blb',''
if collection == 'blb':
  genre_suffix = '_with_genre'

TargetMaskedToken = 'machine' # the token to be masked in the target sentence
ContrastiveMaskedToken = 'slave' # the contrastive token to be masked in the sentence
dataPath = 'masking_data' # change to '.' when working in colab 
processedFolder = 'processed_data' # change '.' when working in colab
predCol = "pred_bert_1760_1900"

print(f"This analysis focuses on '{TargetMaskedToken}' which we constrast to '{ContrastiveMaskedToken}'.")

This analysis focuses on 'machine' which we constrast to 'slave'.


In [3]:
mask2tokens = {'machine':['machine', 'machines'], 'slave':['slave', 'slaves']}

In [ ]:
!unzip -o "/content/{collection}_{TargetMaskedToken}_clusters{genre_suffix}.tsv.zip"
!unzip -o "/content/results_{collection}_{TargetMaskedToken}_constrastive.csv.zip"
!unzip -o "/content/{collection}_{ContrastiveMaskedToken}_clusters{genre_suffix}.tsv.zip"
!unzip -o "/content/results_{collection}_{ContrastiveMaskedToken}_constrastive.csv.zip"

In [4]:
# load the original sentences with the predicted tokens
df_target_sent = pd.read_csv(f'{dataPath}/{collection}_{TargetMaskedToken}_clusters{genre_suffix}.tsv', index_col=0, sep='\t')
print(f'We have {df_target_sent.shape[0]} sentences for the target token {TargetMaskedToken}.')
# load the data with the constrastive explanations
df_target_ig = pd.read_csv(f'{processedFolder}/results_{collection}_{TargetMaskedToken}_constrastive.csv', index_col=0 )
print(f'We have {df_target_ig.shape[0]} explanations for the target token {TargetMaskedToken}.')

We have 131002 sentences for the target token machine.
We have 11008832 explanations for the target token machine.


In [5]:
# load the original sentences with the predicted tokens
df_contrastive_sent = pd.read_csv(f'{dataPath}/{collection}_{ContrastiveMaskedToken}_clusters{genre_suffix}.tsv', index_col=0, sep='\t')
print(f'We have {df_contrastive_sent.shape[0]} sentences for the contrastive token {ContrastiveMaskedToken}.')

# load the data with the constrastive explanations
df_contrastive_ig = pd.read_csv(f'{processedFolder}/results_{collection}_{ContrastiveMaskedToken}_constrastive.csv', index_col=0 )
print(f'We have {df_contrastive_ig.shape[0]} explanations for the contrastive token {ContrastiveMaskedToken}.')

We have 53727 sentences for the contrastive token slave.
We have 4625482 explanations for the contrastive token slave.


In the following cells we inspect the output, you can skip these.

In [6]:
# inspect the data
# each sentence appears twice, once for the original masked token, (Target=maskedtoken) and once for the contrastive token (Target=contrastive)
# i.e. if masked token is 'machine' then contrastive token is 'slave'
df_target_ig.head()

,Token,Score,Target,id
0,the,0.055053,machine,60000
1,[MASK],0.000000,machine,60000
2,shops,0.330447,machine,60000
3,occupy,-0.067776,machine,60000
4,three,0.071201,machine,60000


In [7]:
df_contrastive_ig.head()

,Token,Score,Target,id
0,the,0.090441,machine,0
1,greeks,0.130847,machine,0
2,owe,-0.023468,machine,0
3,their,0.082341,machine,0
4,letters,0.084634,machine,0


In [8]:
# inspect alignment of sentences between dataframes df_sentences and results_df
idx = df_target_sent.shape[0] -2
list(df_target_ig[df_target_ig.id == idx].Token)[:5],df_target_sent.iloc[idx].maskedSentence

(['you', 'would', 'fain', 'be', 'thought'],
 'You would fain be thought to take no share in government , while in reality you are tho main spring of the [MASK] .')

In [9]:
# Note: there is a tiny bug, I forgot the process the last sentence of df_sentences
df_target_ig['id'].nunique(),len(df_target_sent)

(131001, 131002)

# Macro-Scale Influences

At the level of all sentences containing the masked token, which words influence the prediction of observed (or counter-factual) terms.

Why is this interesting?
- **validation**: partly shows that the method works, e.g. the words that influence the prediction of the actual masked token word 'machine' should make sense.
- **counterfactual**: highlight which words in the context of x ('machine') are associated with contrastive term y ('slave'), e.g. which words in the context of machine do contribute to the prediction of slavery, foreground intermingling of two discourses?


## 🛎️ Important 🛎️

In the next cell, decide if you want to analyse the target of contrastive concept.

In [10]:
decision = 'target' # choose whether to analyze the target token or the contrastive token, options: 'target' or 'contrastive'

Based on the `decision` variable we select the correct associated data.

In [14]:
if decision == 'target':
    df_ig = df_target_ig
    df_sent = df_target_sent
elif decision == 'contrastive':
    df_ig = df_contrastive_ig
    df_sent = df_contrastive_sent
else:
    raise ValueError("Invalid decision. Please choose 'target' or 'contrastive'.")
print(f"Analyzing the '{decision}' token '{TargetMaskedToken if decision == 'target' else ContrastiveMaskedToken}': {df_ig.shape[0]} explanations and {df_sent.shape[0]} sentences.")

Analyzing the 'target' token 'machine': 11008832 explanations and 131002 sentences.


In [15]:
# divide results_df into two dataframes, one for the masked token and one for the contrastive token, and calculate the average score for each token in each dataframe
df_observed_ig = df_ig[df_ig['Target'].isin(mask2tokens[TargetMaskedToken])].groupby('Token').agg(avg=('Score', 'mean'), count=('Score', 'size'), std=('Score', 'std'))
df_counterfactual_ig = df_ig[df_ig['Target'].isin(mask2tokens[ContrastiveMaskedToken])].groupby('Token').agg(avg=('Score', 'mean'), count=('Score', 'size'), std=('Score', 'std'))

In [17]:
# just to check, the number rows should be identical
df_observed_ig.shape, df_counterfactual_ig.shape

((74140, 3), (74140, 3))

In [20]:
min_freq = 10 # how often should the token appear in the explanations to be included in the analysis
top_n = 10 # how many tokens to show in the top list
print(f"Top {top_n} tokens with the highest average score for the '{decision}' token '{TargetMaskedToken if decision == 'target' else ContrastiveMaskedToken}'.\nOnly showing tokens that appear at least {min_freq} times in the explanations:")
df_observed_ig[df_observed_ig['count'] > min_freq
                ].sort_values('avg', ascending=False
                              ).head(top_n)[['avg', 'count', 'std']]

Top 10 tokens with the highest average score for the 'target' token 'machine'.
Only showing tokens that appear at least 10 times in the explanations:


,avg,count,std
Token,,,
sewing,0.471022,3870,0.250007
tentering,0.430624,15,0.181372
bathing,0.417719,2452,0.265310
dredging,0.411902,518,0.176709
calculating,0.383839,278,0.235466
electrifying,0.378359,101,0.150086
dling,0.378285,80,0.191377
bureaucratic,0.376194,20,0.103513
administrative,0.374670,79,0.233919


In [21]:
# show the top contributors to the prediction of the masked token 'slave' (or 'slaves')
print(f"Top {top_n} tokens with the highest average score for the '{'contrastive' if decision == 'target' else 'target'}' token '{ContrastiveMaskedToken if decision == 'target' else TargetMaskedToken}'.\nOnly showing tokens that appear at least {min_freq} times in the explanations:")
df_counterfactual_ig[df_counterfactual_ig['count'] > min_freq
                ].sort_values('avg', ascending=False
                              ).head(top_n)[['avg', 'count', 'std']]

Top 10 tokens with the highest average score for the 'contrastive' token 'slave'.
Only showing tokens that appear at least 10 times in the explanations:


,avg,count,std
Token,,,
oratorial,0.358861,18,0.258491
tentering,0.356591,15,0.205735
folard,0.355547,20,0.207538
exploders,0.346915,14,0.245941
wreath,0.340674,14,0.373802
vanes,0.312471,23,0.223666
werder,0.305503,11,0.262240
bureaucratic,0.303477,20,0.118761
infernal,0.291107,1314,0.165619


# Inspect specific token contributions by sentence

Based on these overall contributions, you can select context words and investigate how they interact with slave and machine in concrete sentences.

In [22]:
modelName = "Livingwithmachines/bert_1760_1900"
explainer = MaskedLMExplainer(model_name=modelName, device=pick_device())

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: Livingwithmachines/bert_1760_1900
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
# get the ids for the sentences containing the context token 'tentering' and the masked tokens 'slave' or 'slaves'
context_token = 'tool' 
masked_tokens = mask2tokens[TargetMaskedToken]
print(masked_tokens)
df_ig[(df_ig.Token==context_token) & (df_ig.Target.isin(masked_tokens))]['id'].unique()

['machine', 'machines']


array([ 61362,  61364,  61393,  61394,  61474,  61674,  62580,  64390,
        64396,  64398,  65552,  66090,  69316,  69319,  69322,  69665,
        69671,  69674,  70076,  70083,  70091,  70106,  70109,  70122,
        70126,  70135,  71331,  71500,  71504,  71509,  72667,  72674,
        74249,  74251,  76125,  76589,  76646,  76818,  77315,  77730,
        77731,  77952,  78061,  78470,  78480,  78481,  40281,  40738,
        40740,  43195,  43517,  43518,  44019,  44026,  45032,  45290,
        45580,  45583,  45586,  45661,  45663,  47365,  47713,  47727,
        47735,  47738,  47747,  47755,  47758,  47767,  47770,  47773,
        47775,  47779,  47783,  47784,  47785,  47786,  47792,  47793,
        47794,  47795,  47857,  47858,  47872,  47873,  47880,  47892,
        47893,  47900,  47908,  48062,  48066,  48070,  48988,  49156,
        49158,  49159,  49165,  49167,  49168,  49302,  49317,  49324,
        49475,  49478,  49482,  49796,  49911,  49913,  49914,  49920,
      

In [24]:
# select one of the above ids and highlight the context tokens in the sentence
idx = 1296
sentence = ' '.join(df_ig[(df_ig.id==idx) & df_ig.Target.isin(['slave','slaves'])]['Token'].values)
target = df_ig[(df_ig.id==idx) & df_ig.Target.isin(['slave','slaves'])]['Target'].values[0]
highlight_context_tokens(explainer, sentence, target=target, word_agg="max")

Explaining:   0%|          | 0/1 [00:00<?, ?it/s]

'\n    <div id="tokviz_0ad3498b262c4e6fbe12f5b041bd0da9">\n      <div style=\'margin-bottom:6px;\'>\n        <b>Target:</b> <code>slaves</code>\n      </div>\n      <div style=\'margin:6px 0 10px 0; font-size:13px; display:flex; gap:10px; align-items:center;\'>\n        <span style=\'background:rgba(30,136,229,0.35); padding:2px 8px; border-radius:4px;\'>&#9646; predicts</span>\n        <span style=\'background:rgba(229,57,53,0.35);  padding:2px 8px; border-radius:4px;\'>&#9646; opposes</span>\n        <span style=\'background:rgba(255,193,7,0.85);  padding:2px 8px; border-radius:4px; font-weight:bold;\'>[target] mask position</span>\n      </div>\n      <div style=\'line-height:2.4; font-size:15px;\'>\n        <span class=\'tok\' data-score=\'-0.038385\' style=\'background:rgba(229, 57, 53, 0.180); padding:2px 4px; margin:1px; border-radius:4px; cursor:default;\'>the</span> <span class=\'tok\' data-score=\'-0.009659\' style=\'background:rgba(229, 57, 53, 0.120); padding:2px 4px; margi

# Macro-level differences between actual and counterfactual masked tokens


Previous analysis shows contributions for each masked tokens (actual and counterfactual). However, some words might contribute equally to both masked target words.

Therefore we now focus on differences (the contrast): i.e. which words are most associated with either 'slave' or 'machine'.

In [25]:
df_observed_ig_tok = df_ig[(df_ig['Target'].isin(mask2tokens[TargetMaskedToken]))].reset_index(drop=True)
df_counterfactual_ig_tok = df_ig[(df_ig['Target'].isin(mask2tokens[ContrastiveMaskedToken]))].reset_index(drop=True)

In [29]:
df_observed_ig_tok['Diff_Obs_Counter_Score'] = df_observed_ig_tok['Score'] - df_counterfactual_ig_tok['Score']
df_observed_ig_tok['Counter_Score'] = df_counterfactual_ig_tok['Score']
diff = df_observed_ig_tok.groupby('Token').agg(avg_diff=('Diff_Obs_Counter_Score', 'mean'), count=('Diff_Obs_Counter_Score', 'size'), std_diff=('Diff_Obs_Counter_Score', 'std'), target_scores=('Score', 'mean'), counter_scores=('Counter_Score', 'mean'))#.sort_values('avg_diff', ascending=False).head(10)

In [37]:
print(f"Top {top_n} tokens with the highest average difference in scores between the '{decision}' token '{TargetMaskedToken if decision == 'target' else ContrastiveMaskedToken}' and the '{'contrastive' if decision == 'target' else 'target'}' token '{ContrastiveMaskedToken if decision == 'target' else TargetMaskedToken}'.\nOnly showing tokens that appear at least {min_freq} times in the explanations:")
print("Showing the words most leaning to '{}'.".format(TargetMaskedToken if decision == 'target' else ContrastiveMaskedToken))
diff[diff['count'] > 10].sort_values('avg_diff', ascending=False).head(top_n)

Top 10 tokens with the highest average difference in scores between the 'target' token 'machine' and the 'contrastive' token 'slave'.
Only showing tokens that appear at least 10 times in the explanations:
Showing the words most leaning to 'machine'.


,avg_diff,count,std_diff,target_scores,counter_scores
Token,,,,,
sewing,0.382744,3870,0.305014,0.471022,0.088278
bathing,0.357056,2452,0.278600,0.417719,0.060663
dredging,0.317491,518,0.189366,0.411902,0.094411
crashing,0.286923,48,0.330577,0.198891,-0.088033
administrative,0.282670,79,0.244242,0.374670,0.092000
calculating,0.272053,278,0.256406,0.383839,0.111786
exploding,0.267142,25,0.289629,0.126354,-0.140787
filtering,0.254285,32,0.217420,0.328556,0.074270
spiked,0.251594,16,0.295239,0.075453,-0.176141


In [38]:
print("Showing the words most leaning to '{}'.".format(ContrastiveMaskedToken if decision == 'target' else TargetMaskedToken))
diff[diff['count'] > 10].sort_values('avg_diff', ascending=True).head(top_n)

Showing the words most leaning to 'slave'.


,avg_diff,count,std_diff,target_scores,counter_scores
Token,,,,,
arabian,-0.259229,16,0.183309,-0.003792,0.255436
superiors,-0.221889,16,0.215575,-0.043352,0.178537
familiarity,-0.199497,11,0.252397,0.009573,0.209070
tucker,-0.184309,22,0.319949,-0.072574,0.111735
roughing,-0.180935,19,0.176782,0.107167,0.288103
wreath,-0.178590,14,0.148884,0.162084,0.340674
borrowing,-0.163382,15,0.123589,-0.044231,0.119151
gallantry,-0.161409,11,0.169348,-0.061494,0.099915
beech,-0.160942,11,0.407797,-0.019432,0.141511


# Sentence-level differences

For which sentences do we observe the largest shift in context contribution when switching between different target words?

Shifts are often largest for shorter sentences. Therefore, I added threshold to include only sentence above a certain tokens length.

In [39]:
diff_dict = {'slave':'machine', 'slaves':'machines','machine':'slave', 'machines':'slaves'}


In [42]:
min_length = 20 # threshold to filter out sentences that are too short 
top_n = 10 # show the top n sentences with the highest average difference in scores between the masked token and the contrastive token
by_sent = df_observed_ig_tok.groupby('id').agg(avg_diff=('Diff_Obs_Counter_Score', 'mean'), count=('Diff_Obs_Counter_Score', 'size'))#
print(f"Top {top_n} sentences with the highest average difference in scores when changing the mask from the '{decision}' token '{TargetMaskedToken if decision == 'target' else ContrastiveMaskedToken}' and the '{'contrastive' if decision == 'target' else 'target'}' token '{ContrastiveMaskedToken if decision == 'target' else TargetMaskedToken}'.\nOnly showing sentences that have at least {min_length} tokens in the explanations:")
by_sent[by_sent['count'] > min_length].sort_values('avg_diff', ascending=False).head(top_n)

Top 10 sentences with the highest average difference in scores when changing the mask from the 'target' token 'machine' and the 'contrastive' token 'slave'.
Only showing sentences that have at least 20 tokens in the explanations:


,avg_diff,count
id,,
118711,0.211902,24
2257,0.198047,22
2236,0.198047,22
108511,0.184913,25
108513,0.184913,25
18843,0.169937,22
129871,0.156638,25
129862,0.156638,25
129420,0.156638,25


In [ ]:
modelName = "Livingwithmachines/bert_1760_1900"
explainer = MaskedLMExplainer(model_name=modelName, device=pick_device())

In [ ]:
idx = 13274 # select one of the adove ids with a high average difference in scores between the masked token and the contrastive token
sentence = ' '.join(df_ig[(df_ig.id==idx) & df_ig.Target.isin(mask2tokens[TargetMaskedToken])]['Token'].values)
target = df_ig[(df_ig.id==idx) & df_ig.Target.isin(mask2tokens[TargetMaskedToken])]['Target'].values[0]
#print(sentence, target)
highlight_context_tokens(explainer, sentence, target=target, word_agg="max")

In [ ]:
highlight_context_tokens(explainer, sentence, target=diff_dict[TargetMaskedToken], word_agg="max")

# Overlapping contributors

Which words are strong predictors of the actual and counterfactual token?

Here we look at the top_n contributes for both 'machine' and 'slave' (which appear more than n_count times in the context of the maskedToken) and take the overlap between the top ranked predictors.

In [43]:
min_count = 25 # how often should token appear in the explanations to be included in the analysis of the top contributing tokens to the prediction of the masked token
top_n = 100 # look at top n contrastive tokens
top_observed_indices = df_observed_ig[df_observed_ig['count'] > min_count].sort_values('avg', ascending=False).head(top_n).index
top_counterfactual_indices = df_counterfactual_ig[df_counterfactual_ig['count'] > min_count].sort_values('avg', ascending=False).head(top_n).index
overlap_indices = top_observed_indices.intersection(top_counterfactual_indices)

In [51]:
print(f"Overlap among the top {top_n} contributors for the '{TargetMaskedToken}' and '{ContrastiveMaskedToken}' with at least {min_count} occurrences in the explanations:")
overlap_indices

Overlap among the top 10 contributors for the 'machine' and 'slave' with at least 10 occurrences in the explanations:


Index(['electrifying', 'dling', 'slaying', 'threshing', 'soulless', 'guns',
       'pendulum', 'electrical', 'jigging', 'cone', 'photographic', 'gun',
       'political', 'copying', 'mowing', 'plug', 'camera', 'glove', 'ry',
       'infernal', 'organism', 'blackwood', 'crane', 'kneading', 'winnowing',
       'praying', 'reaping', 'riddle', 'unoiled', 'pan', 'machines', 'drill',
       'mere', 'trical', 'stump', 'trough', 'feat', 'compartment', 'fans',
       'vanners', 'machine', 'une', 'frame'],
      dtype='str', name='Token')

# Using Metadata and Cluster Scores

## Time

Here we focus on sentence published between start_year and end_year

In [48]:
start_year = 1800
end_year = 1810
selected_sentences = df_sent[df_sent['date'].between(start_year, end_year)].index

In [49]:
df_observed_ig_sel = df_ig[(df_ig['Target'].isin(mask2tokens[TargetMaskedToken])) & (df_ig['id'].isin(selected_sentences))].groupby('Token').agg(avg=('Score', 'mean'), count=('Score', 'size'), std=('Score', 'std'))
df_counterfactual_ig_sel = df_ig[(df_ig['Target'].isin(mask2tokens[ContrastiveMaskedToken])) & (df_ig['id'].isin(selected_sentences))].groupby('Token').agg(avg=('Score', 'mean'), count=('Score', 'size'), std=('Score', 'std'))


In [52]:
min_count = 10 # how often should token appear in the explanations to be included in the analysis of the top contributing
top_n = 10
print(f"Top {top_n} tokens with the highest average score for the '{ContrastiveMaskedToken if decision == 'target' else TargetMaskedToken}' in sentences from {start_year} to {end_year}.\nOnly showing tokens that appear at least {min_count} times in the explanations:")
df_counterfactual_ig_sel[df_counterfactual_ig_sel['count'] > min_count
                ].sort_values('avg', ascending=False
                              ).head(top_n)[['avg', 'count', 'std']]

Top 10 tokens with the highest average score for the 'slave' in sentences from 1800 to 1810.
Only showing tokens that appear at least 10 times in the explanations:


,avg,count,std
Token,,,
hydraulic,0.362136,38,0.206643
infernal,0.311885,19,0.189531
electrical,0.289307,17,0.176692
pump,0.279499,14,0.225323
threshing,0.277673,19,0.240998
ry,0.263159,11,0.286471
pan,0.259594,11,0.231331
mere,0.234311,59,0.218953
clocks,0.219860,12,0.256600


In [53]:
print(f"Top {top_n} tokens with the highest average score for the '{decision}' token '{TargetMaskedToken if decision == 'target' else ContrastiveMaskedToken}' in sentences from {start_year} to {end_year}.\nOnly showing tokens that appear at least {min_count} times in the explanations:")
df_observed_ig_sel[df_observed_ig_sel['count'] > min_count
                ].sort_values('avg', ascending=False
                              ).head(top_n)[['avg', 'count', 'std']]

Top 10 tokens with the highest average score for the 'target' token 'machine' in sentences from 1800 to 1810.
Only showing tokens that appear at least 10 times in the explanations:


,avg,count,std
Token,,,
electrical,0.386584,17,0.180264
dredging,0.384913,22,0.152761
shaft,0.376188,12,0.308198
hydraulic,0.342299,38,0.221731
pump,0.331008,14,0.281246
political,0.313466,46,0.217951
pan,0.307325,11,0.234481
engines,0.305669,33,0.214215
threshing,0.299179,19,0.176211


## Cluster-based filtering

Let's focus only on the sentence that scored high for the 'slave' cluster in the df_sentences dataframe.

In [59]:
top_cl_sent = 1000
colName = 'slave_1760_1900'
selected_sentences = df_sent.sort_values(by=colName, ascending=False)[:top_cl_sent].index

In [60]:
df_target_ig_sel = df_ig[(df_ig['Target'].isin(mask2tokens[TargetMaskedToken])) & (df_ig['id'].isin(selected_sentences))].groupby('Token').agg(avg=('Score', 'mean'), count=('Score', 'size'), std=('Score', 'std'))
df_counterfactual_ig_sel = df_ig[(df_ig['Target'].isin(mask2tokens[ContrastiveMaskedToken])) & (df_ig['id'].isin(selected_sentences))].groupby('Token').agg(avg=('Score', 'mean'), count=('Score', 'size'), std=('Score', 'std'))


In [61]:
min_count = 10 # how often should token appear in the explanations to be included in the analysis of the top contributing
top_n = 10
print(f"Top {top_n} tokens with the highest average score for the '{ContrastiveMaskedToken if decision == 'target' else TargetMaskedToken}' in the top {top_cl_sent} sentences with the highest scores in '{colName}'.\nOnly showing tokens that appear at least {min_count} times in the explanations:")
df_counterfactual_ig_sel[df_counterfactual_ig_sel['count'] > min_count
                ].sort_values('avg', ascending=False
                              ).head(top_n)[['avg', 'count', 'std']]

Top 10 tokens with the highest average score for the 'slave' in the top 1000 sentences with the highest scores in 'slave_1760_1900'.
Only showing tokens that appear at least 10 times in the explanations:


,avg,count,std
Token,,,
servants,0.300256,22,0.227196
mere,0.299474,137,0.177209
soldier,0.288940,15,0.235589
beings,0.286983,18,0.243868
slave,0.261036,33,0.191857
hundred,0.256948,23,0.194793
human,0.251165,84,0.239230
work,0.236881,173,0.233282
soldiers,0.234227,17,0.159183


In [64]:
print(f"Top {top_n} tokens with the highest average score for the '{decision}' token '{TargetMaskedToken if decision == 'target' else ContrastiveMaskedToken}' in the top {top_cl_sent} sentences with the highest scores in '{colName}'.\nOnly showing tokens that appear at least {min_count} times in the explanations:")
df_observed_ig_sel[df_observed_ig_sel['count'] > min_count
                ].sort_values('avg', ascending=False
                              ).head(top_n)[['avg', 'count', 'std']]

Top 10 tokens with the highest average score for the 'target' token 'machine' in the top 1000 sentences with the highest scores in 'slave_1760_1900'.
Only showing tokens that appear at least 10 times in the explanations:


,avg,count,std
Token,,,
electrical,0.386584,17,0.180264
dredging,0.384913,22,0.152761
shaft,0.376188,12,0.308198
hydraulic,0.342299,38,0.221731
pump,0.331008,14,0.281246
political,0.313466,46,0.217951
pan,0.307325,11,0.234481
engines,0.305669,33,0.214215
threshing,0.299179,19,0.176211


# Fin.